In [ ]:
pip install -U langchain


In [ ]:
pip install -U langchain-google-genai

In [ ]:
from typing import TypedDict, Annotated, List, Dict, Any
import operator
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata




llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("gemini_api_key")
)




class InterviewState(TypedDict):
    messages: Annotated[list, operator.add]
    job_description: str
    candidate_info: str
    jd_analysis: str
    experience_mapping: str
    behavioral_evaluation: str
    technical_evaluation: str
    final_report: str


In [ ]:
def jd_analyst_node(state: InterviewState):
    prompt = f"""
You are a JD Analyst Agent.


Analyze this job description and extract:
1. Role title
2. Required skills
3. Responsibilities
4. Experience level
5. Important keywords
6. What the interviewer may focus on


Job Description:
{state['job_description']}
"""
    response = llm.invoke([HumanMessage(content=prompt)])


    return {
        "jd_analysis": response.content,
        "messages": [AIMessage(content=f"JD Analyst Completed:\n{response.content}")]
    }

In [ ]:
def experience_mapper_node(state: InterviewState):
    prompt = f"""
You are an Experience Mapper Agent.


Compare the candidate information with the JD analysis.


Candidate Info:
{state['candidate_info']}


JD Analysis:
{state['jd_analysis']}


Find:
1. Candidate strengths
2. Skill gaps
3. Projects/experiences that match the JD
4. How candidate should present themselves
5. 2 personalized experience-based interview questions
"""


    response = llm.invoke([HumanMessage(content=prompt)])


    return {
        "experience_mapping": response.content,
        "messages": [AIMessage(content=f"Experience Mapper Completed:\n{response.content}")]
    }


In [ ]:
def behavioral_interviewer_node(state: InterviewState):
    prompt = f"""
You are a Behavioral Interview Agent.


Based on the candidate profile and JD, generate:
1. Two behavioral interview questions
2. What a good answer should include
3. Common mistakes candidate may make


Candidate Info:
{state['candidate_info']}


JD Analysis:
{state['jd_analysis']}


Experience Mapping:
{state['experience_mapping']}
"""


    response = llm.invoke([HumanMessage(content=prompt)])


    return {
        "behavioral_evaluation": response.content,
        "messages": [AIMessage(content=f"Behavioral Interview Agent Completed:\n{response.content}")]
    }


In [ ]:
def technical_interviewer_node(state: InterviewState):
    prompt = f"""
You are a Technical Interview Agent.


Based on the JD, generate:
1. Three role-specific technical interview questions
2. Expected answer points
3. Mistakes or blunders to avoid


JD Analysis:
{state['jd_analysis']}
Candidate Info:
{state['candidate_info']}
"""


    response = llm.invoke([HumanMessage(content=prompt)])


    return {
        "technical_evaluation": response.content,
        "messages": [AIMessage(content=f"Technical Interview Agent Completed:\n{response.content}")]
    }




def brain_evaluator_node(state: InterviewState):
    prompt = f"""
You are the Brain Evaluator Agent.


Prepare a final interview preparation report using all previous agent outputs.


Candidate Info:
{state['candidate_info']}


JD Analysis:
{state['jd_analysis']}


Experience Mapping:
{state['experience_mapping']}


Behavioral Evaluation:
{state['behavioral_evaluation']}


Technical Evaluation:
{state['technical_evaluation']}


Final report must include:
1. Overall candidate readiness score out of 100
2. JD match score
3. Communication readiness
4. Technical readiness
5. Strengths
6. Weak areas
7. Mistakes to avoid
8. Serious blunders to avoid
9. Final preparation strategy
"""


    response = llm.invoke([HumanMessage(content=prompt)])


    return {
        "final_report": response.content,
        "messages": [AIMessage(content=f"Brain Evaluator Completed:\n{response.content}")]
    }


In [ ]:
workflow = StateGraph(InterviewState)


workflow.add_node("jd_analyst", jd_analyst_node)
workflow.add_node("experience_mapper", experience_mapper_node)
workflow.add_node("behavioral_interviewer", behavioral_interviewer_node)
workflow.add_node("technical_interviewer", technical_interviewer_node)
workflow.add_node("brain_evaluator", brain_evaluator_node)




workflow.set_entry_point("jd_analyst")


workflow.add_edge("jd_analyst", "experience_mapper")
workflow.add_edge("experience_mapper", "behavioral_interviewer")
workflow.add_edge("behavioral_interviewer", "technical_interviewer")
workflow.add_edge("technical_interviewer", "brain_evaluator")
workflow.add_edge("brain_evaluator", END)




app = workflow.compile()

In [ ]:
job_description = input("Paste the Job Description:\n")
candidate_info = input("\nEnter candidate skills, projects, experience:\n")


initial_state = {
    "messages": [],
    "job_description": job_description,
    "candidate_info": candidate_info,
    "jd_analysis": "",
    "experience_mapping": "",
    "behavioral_evaluation": "",
    "technical_evaluation": "",
    "final_report": ""
}


result = app.invoke(initial_state)


print("\nFINAL INTERVIEW REPORT\n")
print(result["final_report"])
